In [8]:
#pip install azure-identity azure-ai-projects openai

# Crie um aplicativo de bate-papo com IA generativa.

Neste exercício, você usará o SDK do Azure AI Foundry para Python para criar um aplicativo de bate-papo simples que se conecta a um projeto e conversa com um modelo de linguagem.

## Implantar um modelo em um projeto do Azure AI Foundry

Vamos começar implantando um modelo em um projeto do Azure AI Foundry.

1. Em um navegador da Web, abra o portal do Azure AI Foundry em https://ai.azure.com e faça login usando suas credenciais do Azure. Feche quaisquer dicas ou painéis de início rápido que sejam abertos na primeira vez que você fizer login e, se necessário, use o logotipo do Azure AI Foundry no canto superior esquerdo para navegar até a página inicial, que é semelhante à imagem a seguir (feche o painel de Ajuda , se estiver aberto):

2. Na página inicial, na seção Explorar modelos e recursos , procure o `gpt-4o` modelo que usaremos em nosso projeto.

3. Nos resultados da pesquisa, selecione o modelo `gpt-4o` para ver os detalhes e, em seguida, na parte superior da página do modelo, selecione `Usar este modelo` .

4. Quando solicitado a criar um projeto, insira um nome válido para o seu projeto e expanda as `Opções avançadas` .

5. Selecione `Personalizar` e especifique as seguintes configurações para o seu projeto:
    - Recurso do Azure AI Foundry : Um nome válido para o seu recurso do Azure AI Foundry.
    - Assinatura : Sua assinatura do Azure
    - Grupo de recursos : Crie ou selecione um grupo de recursos.
    - Região : Selecione qualquer região recomendada pela AI Foundry *

6. Selecione `Criar` e aguarde a criação do seu projeto, incluindo a implantação do modelo gpt-4o que você selecionou.

7. Quando seu projeto for criado, o ambiente de testes de bate-papo será aberto automaticamente para que você possa testar seu modelo (caso contrário, no painel de tarefas à esquerda, selecione `Ambientes de Teste` e abra o ambiente de `testes de bate-papo` ).

8. No painel `de Configuração` , observe o nome da sua implantação do modelo; que deve ser `gpt-4o` . Você pode confirmar isso visualizando a implantação na página `Modelos e endpoints` (basta abrir essa página no painel de navegação à esquerda).

## Crie um aplicativo cliente para conversar com a modelo.

Agora que você implantou um modelo, pode usar os SDKs do Azure AI Foundry e do Azure OpenAI para desenvolver um aplicativo que interaja com ele.

### Prepare a configuração do aplicativo.

1. No portal do Azure AI Foundry, visualize a página `Visão geral` do seu projeto.

2. Na área `Pontos de extremidade e chaves` , certifique-se de que a biblioteca `Azure AI Foundry esteja selecionada e visualize o ponto de extremidade do projeto Azure AI Foundry` . Você usará esse ponto de extremidade para se conectar ao seu projeto e modelo em um aplicativo cliente.

> **Observação** : Você também pode usar o endpoint do Azure OpenAI!

3. Abra uma nova aba do navegador (mantendo o portal do Azure AI Foundry aberto na aba atual). Em seguida, na nova aba, acesse o portal do Azure em https://portal.azure.com ; faça login com suas credenciais do Azure, se solicitado.

4. Use o botão `[>_]` à direita da barra de pesquisa na parte superior da página para criar um novo Cloud Shell no portal do Azure, selecionando um ambiente `do PowerShell` sem armazenamento em sua assinatura.

O Cloud Shell fornece uma interface de linha de comando em um painel na parte inferior do portal do Azure. Você pode redimensionar ou maximizar esse painel para facilitar o trabalho.

> **Observação** : Se você já criou um shell na nuvem que usa um ambiente Bash , mude para PowerShell .

5. Na barra de ferramentas do Cloud Shell, no menu `Configurações` , selecione `Ir para a versão clássica` (isso é necessário para usar o editor de código).

6. No painel do Cloud Shell, insira os seguintes comandos para clonar o repositório do GitHub que contém os arquivos de código para este exercício (digite o comando ou copie-o para a área de transferência e, em seguida, clique com o botão direito do mouse na linha de comando e cole como texto sem formatação):

```powershell
    rm -r mslearn-ai-foundry -f
    git clone https://github.com/microsoftlearning/mslearn-ai-studio mslearn-ai-foundry
```

> **Dica** : Ao inserir comandos no cloudshell, a saída pode ocupar uma grande parte do buffer da tela. Você pode limpar a tela digitando o `cls` comando para facilitar o foco em cada tarefa.

7. Após clonar o repositório, navegue até a pasta que contém os arquivos de código do aplicativo de bate-papo e visualize-os:

```powershell
    cd mslearn-ai-foundry/labfiles/chat-app/python
    ls -a -l
```

A pasta contém um arquivo de código, bem como um arquivo de configuração para as definições da aplicação e um arquivo que define o ambiente de execução do projeto e os requisitos do pacote.

8. No painel de linha de comando do Cloud Shell, digite o seguinte comando para instalar as bibliotecas que você usará:

```powershell
    python -m venv labenv
    ./labenv/bin/Activate.ps1
    pip install -r requirements.txt azure-identity azure-ai-projects openai
```

9. Digite o seguinte comando para editar o arquivo de configuração fornecido:

```powershell
    code .env
```

O arquivo é aberto em um editor de código.

> **Importante** : Se você implantou seu modelo gpt-4o na região padrão do seu projeto, pode usar o ponto de extremidade do `projeto do Azure AI Foundry` ou do `Azure OpenAI` na página `Visão geral` do seu projeto para se conectar ao seu modelo. Se você não tinha cota suficiente e implantou o modelo em outra região, na página `Modelos + Pontos de extremidade` , selecione seu modelo e use o `URI de destino` do seu modelo.

10. No arquivo de código, substitua o marcador `your_project_endpoint` pelo endpoint apropriado para o seu modelo e o marcador `your_model_deployment` pelo nome atribuído à sua implantação do modelo gpt-4o.

11. Após substituir os marcadores de posição, no editor de código, use o comando `CTRL+S` ou clique com o botão direito do mouse > Salvar para salvar as alterações e, em seguida, use o comando `CTRL+Q` ou clique com o botão direito do mouse > Sair para fechar o editor de código, mantendo a linha de comando do Cloud Shell aberta.

### Escreva o código para se conectar ao seu projeto e conversar com seu modelo.

> **Dica** : Ao adicionar código, certifique-se de manter a indentação correta.

1. Digite o seguinte comando para editar o arquivo de código fornecido:

```powershell
    code chat-app.py
```

2. No arquivo de código, observe as instruções adicionadas no início do arquivo para importar os namespaces do SDK necessários. Em seguida, localize o comentário "**Adicionar referências**" e adicione o seguinte código para referenciar os namespaces nas bibliotecas que você instalou anteriormente:

In [ ]:
# Add references
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from openai import AzureOpenAI

3. Na função **principal** , sob o comentário "**Obter configurações**" , observe que o código carrega os valores da string de conexão do projeto e do nome de implantação do modelo que você definiu no arquivo de configuração.

4. Localize o comentário "**Inicialize the project client**" e adicione o seguinte código para conectar-se ao seu projeto do Azure AI Foundry:

> **Dica** : Tenha cuidado para manter o nível de indentação correto no seu código

In [ ]:
project_endpoint = ""

# Initialize the project client
project_client = AIProjectClient(            
         credential=DefaultAzureCredential(
             exclude_environment_credential=True,
             exclude_managed_identity_credential=True
         ),
         endpoint=project_endpoint,
     )

5. Localize o comentário "**Obter um cliente de bate-papo**" e adicione o seguinte código para criar um objeto cliente para conversar com um modelo:

In [ ]:
# Get a chat client
openai_client = project_client.get_openai_client(api_version="2024-10-21")

6. Localize o comentário "**Initialize prompt with system message**" e adicione o seguinte código para inicializar uma coleção de mensagens com um prompt do sistema.

In [ ]:
# Initialize prompt with system message
prompt = [
    {
        "role": "system", 
        "content": "You are a helpful AI assistant that answers questions."
    }
]

7. Observe que o código inclui um loop para permitir que o usuário insira uma mensagem até digitar "sair". Em seguida, na seção do loop, encontre o comentário "**Obter uma conclusão do chat**" e adicione o seguinte código para adicionar a entrada do usuário à mensagem, recuperar a conclusão do seu modelo e adicionar a conclusão à mensagem (para que você mantenha o histórico do chat para iterações futuras):

In [ ]:
input_text="""
"""

In [ ]:
model_deployment = ""

# Get a chat completion
prompt.append({"role": "user", "content": input_text})

response = openai_client.chat.completions.create(
         model=model_deployment,
         messages=prompt)

completion = response.choices[0].message.content

print(completion)

prompt.append({"role": "assistant", "content": completion})

8. Use o comando **CTRL+S** para salvar as alterações no arquivo de código.

### Inicie sessão no Azure e execute a aplicação.

1. No painel de linha de comando do Cloud Shell, digite o seguinte comando para entrar no Azure.


```powershell

    az login

```


**Você precisa fazer login no Azure, mesmo que a sessão do Cloud Shell já esteja autenticada.**

2. Quando solicitado, siga as instruções para abrir a página de login em uma nova guia e insira o código de autenticação fornecido e suas credenciais do Azure. Em seguida, conclua o processo de login na linha de comando, selecionando a assinatura que contém seu hub do Azure AI Foundry, se solicitado.

3. Após efetuar o login, digite o seguinte comando para executar o aplicativo:


```powershell

    python chat-app.py 

```


4. Quando solicitado, insira uma pergunta, como por exemplo `What is the fastest animal on Earth?`, e revise a resposta do seu modelo de IA generativa.

5. Experimente fazer algumas perguntas de acompanhamento, como `Where can I see one?` ou `Are they endangered?`. A conversa deve continuar, usando o histórico do chat como contexto para cada interação.

6. Ao terminar, pressione Enter `quit` para sair do programa.

## Resumo
Neste exercício, você usou o SDK do Azure AI Foundry para criar um aplicativo cliente para um modelo de IA generativa que você implantou em um projeto do Azure AI Foundry.

## Limpar
Se você já terminou de explorar o portal do Azure AI Foundry, deve excluir os recursos que criou neste exercício para evitar custos desnecessários do Azure.

1. Abra o `portal do Azure` e visualize o conteúdo do grupo de recursos onde você implantou os recursos usados ​​neste exercício.
2. Na barra de ferramentas, selecione **Excluir grupo de recursos** .
3. Digite o nome do grupo de recursos e confirme que deseja excluí-lo.